In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Standard imports
import numpy as np
import xarray as xr
import tqdm as tqdm

# For variogram estimation
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage import label
from scipy.spatial.distance import pdist
from scipy.optimize import curve_fit

In [3]:
# Import OpenSense modules as submodules
import sys
import os

sys.path.append(os.path.abspath("./pycomlink/"))
sys.path.append(os.path.abspath("./poligrain/src/"))
sys.path.append(os.path.abspath("./mergeplg/src/"))

import pycomlink as pycml 
import poligrain as plg
import mergeplg 

In [4]:
os.makedirs('data/metrics', exist_ok=True)

In [5]:
# Define function to estimate variogram from rainfall event
def get_event_variogram(da_event, bin_edges, min_obs, plot_variogram=False):
    """
    da_event: xarray data for computing variogram
    bin_edges: array of distance bins [m] (e.g., np.linspace(0, 30000, 1500))
    min_obs: minimum observations needed to perform variogram estimation
    plot_variogram: whether to plot the variogram
    """
    all_distances = []
    all_sq_diffs = []
    
    # 1. Collect pairs from all time steps in the event
    n_obs = 0
    for t in da_event.time:
        # Extract data for this timestamp and drop nan
        data_t = da_event.sel(time=t).dropna(dim='cml_id')
        
        # We need at least 2 points to make a pair
        if len(data_t.cml_id) < 2:
            continue

        n_obs += data_t.cml_id.size

        # Get coordinates of data
        coords = np.column_stack([data_t.x, data_t.y])
        values = data_t.values
        
        # Calculate distances and 0.5 * (zi - zj)^2
        dist = pdist(coords, metric='euclidean')

        # Calculate squared distance (variance)
        sq_diff = 0.5 * pdist(values[:, None], 'sqeuclidean')
        
        all_distances.append(dist)
        all_sq_diffs.append(sq_diff)

    # If not enough observations
    if n_obs < min_obs:
        return None
        
    # Flatten into two long arrays of all pairs in the event
    dist_pool = np.concatenate(all_distances)
    diff_pool = np.concatenate(all_sq_diffs)
    
    # 2. Binning 
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    gamma_obs = []
    count = []

    # Also drop largest outliers
    #diff_upper = np.nanquantile(dist_pool, q = 0.95)
    
    for i in range(len(bin_edges)-1):
        mask = (dist_pool > bin_edges[i]) & (dist_pool <= bin_edges[i+1]) # & (diff_pool <= diff_upper) 
        if np.any(mask):
            gamma_obs.append(np.mean(diff_pool[mask]))
            count.append(diff_pool[mask].size)
        else:
            gamma_obs.append(np.nan)
            count.append(0)

    count = np.array(count)
    gamma_obs = np.array(gamma_obs)

    # 3. Define valid observation bins and get variance
    valid = ~np.isnan(gamma_obs) & (count != 0) 

    # Locate where most observations are
    dist_lower = np.nanquantile(dist_pool, q = 0.05)
    dist_upper = np.nanquantile(dist_pool, q = 0.95)
    ind_lower = np.where(dist_lower < bin_centers[valid])[0][0]

    if (dist_upper > bin_centers[valid]).all():
        ind_upper = bin_centers[valid].size
    else:
        ind_upper = np.where(dist_upper < bin_centers[valid])[0][0]

    total_variance = np.nanmean(gamma_obs[valid][ind_lower:ind_upper])

    # If no variance 
    if total_variance == 0: 
        return None

    gamma_obs_norm = gamma_obs/total_variance

    # 4. Define variogram and bounds
    def spherical_model(h, nugget, p_sill, range_a):
        # Use same definitions as pykrige:
        # https://geostat-framework.readthedocs.io/projects/pykrige/en/stable/variogram_models.html
        return np.where(h <= range_a, 
                        p_sill * (1.5 * (h/range_a) - 0.5 * (h/range_a)**3) + nugget, 
                        p_sill + nugget)
        
    # Estimate partial sill from normalized variance
    def f(h, r):
        return spherical_model(h, 0.2, 0.8, r)
        
    # Initial guess: [min(gamma), max_dist/2]
    p0 = [np.max(bin_centers[valid][ind_lower:ind_upper])/2]

    # Parameter bounds
    bound_l_range = bin_centers[valid][ind_lower]
    bound_u_range = np.max(bin_edges)*2
    
    bounds = [bound_l_range, bound_u_range]
    
    # 5. Optimize and return parameters
    popt, _ = curve_fit(
        f, 
        bin_centers[valid][:ind_upper], 
        gamma_obs_norm[valid][:ind_upper], 
        p0=p0, 
        bounds=bounds,
    )

    nugget = 0.2
    p_sill = 1 - nugget
    range_a = popt[0]

    if plot_variogram:
        fig, ax = plt.subplots(1, 1)
        ax.plot(bin_centers[valid], spherical_model(bin_centers[valid], nugget, p_sill, range_a), label='plot variogram')
        ax.plot(bin_centers[valid][:ind_upper], gamma_obs_norm[valid][:ind_upper], label='data used')
        ax.plot(bin_centers[valid][ind_upper:], gamma_obs_norm[valid][ind_upper:], label='data left out')
        plt.legend()
        plt.show()
    
    return n_obs, [nugget, p_sill, range_a]


In [6]:
def estimate_several_variograms(ds_cmls, target_variable):
    # Variogram difference radar-cml
    bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
    min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search
    
    # Group nearby rainfall (closer than 6 hours) to estimate variogram
    mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
    mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
    labels_raw, num_features = label(mask)
    labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)
    
    variograms = []
    # Estimate variogram for rainfall events
    for i in tqdm.tqdm(range(1, num_features + 1)):
        # neighbouring events to include
        n_neighbors = 0 
    
        # Event for variogram estimation
        target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
        combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
        # Expand neighbourhood until we have enough data
        expand = True
        while expand:
            # Get start-end time
            time_start = combined_event_time.time.values[0]
            time_end = combined_event_time.time.values[-1]
            time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
            
            # Estimate variogram 
            out = get_event_variogram( # Here nugget fix 0.2
                ds_cmls[target_variable].sel(time = slice(time_start, time_end)), # 
                bin_edges, 
                min_obs,
                plot_variogram = False,
            )
    
            # If variogram estimation was successful, stop expanding
            if out is not None:
                expand = False 
                n_obs = out[0]
                variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
                
            else:
                # Timesteps included in event
                n_time = combined_event_time.time.size
    
                # Expand neighborhood
                n_neighbors +=1
                target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
                combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
        
                # Break loop if exand did not result in more timesteps
                if n_time >= combined_event_time.time.size:
                    expand = False
                    print('Warning: Not enough rainfall obs in time series, using standard instead')
                    variograms = [[time_mid, 0.2, 0.8, 30000]]
    
    variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'psill', 'range']).set_index('time')
    
    # Aggregate variogram parameters across timesteps
    variograms = variograms.resample('14D', label='left').mean()
    
    # Add first and last timestep of CML to variogram dataframe, completing the time series
    start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
    end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
    variograms = pd.concat([start_row, variograms, end_row]).sort_index()
    return variograms

In [7]:
# Evaluation fucntion
def compute_metrics(field, field_name, dataset):
    # RAINFALL FIELDS AT THE RAIN GAUGES
    get_grid_at_points = plg.spatial.GridAtPoints(
        da_gridded_data=ds_rad.isel(time = 0), 
        da_point_data=ds_gauges.isel(time = 0),
        nnear=1,
        stat="best" 
    )

    ds_gauges[field_name] = get_grid_at_points(
        da_gridded_data=field,
        da_point_data=ds_gauges.rainfall_amount,  
    )

    # COMPUTE METRICS
    threshold = 0.2

    metric = pd.DataFrame([plg.validation.calculate_rainfall_metrics(
        reference=ds_gauges.rainfall_amount.values.flatten(),
        estimate=ds_gauges[field_name].values.flatten(),
        ref_thresh=threshold,
        est_thresh=threshold,
    )]) 
    metric['dataset'] = dataset
    metric['method'] = field_name

    metric.to_csv(field_name)

# OpenMRG adjustment

In [8]:
# OpenMRG
ds_rad = xr.open_dataset("data/andersson_2022_OpenMRG/radar/openmrg_rad.nc")                    
ds_cmls = xr.open_dataset("data/processed_cml_OpenMRG.nc")       
ds_gauges = xr.open_dataset('data/andersson_2022_OpenMRG/gauges/openmrg_gauges.nc')     

# Get radar along CML, used for variogram estimation
da_intersect_weights = plg.spatial.calc_sparse_intersect_weights_for_several_cmls(
    x1_line=ds_cmls.site_0_lon.values,
    y1_line=ds_cmls.site_0_lat.values,
    x2_line=ds_cmls.site_1_lon.values,
    y2_line=ds_cmls.site_1_lat.values,
    cml_id=ds_cmls.cml_id.values,
    x_grid=ds_rad.lon.values,
    y_grid=ds_rad.lat.values,
    grid_point_location='center',
)
ds_cmls['radar_along_cml'] = plg.spatial.get_grid_time_series_at_intersections(
    grid_data=ds_rad.rainfall_amount,
    intersect_weights=da_intersect_weights,
)

# Difference used for additive
ds_cmls['rainfall_difference'] = xr.where(
    ds_cmls.radar_along_cml > 0, 
    ds_cmls.R_acc - ds_cmls.radar_along_cml, 
    np.nan
)

# Ratio used for multiplicative
ds_cmls['rainfall_ratio'] = xr.where(
    ds_cmls.radar_along_cml > 0, 
    ds_cmls.R_acc/ds_cmls.radar_along_cml, 
    np.nan
)

# CML obs used for KED
ds_cmls['rainfall_cml'] = xr.where(
    ds_cmls.radar_along_cml > 0, 
    ds_cmls.R_acc, 
    np.nan
)

# For IDW loop
months = ['2015-06', '2015-07', '2015-08']                           

In [9]:
# methods parameter : default version
ratio_check_sel = (0.1,15)
diff_check_sel=10
nnears = [12, 70, 500] # 500 is more than all

In [10]:
# additive IDW 

for nnear in nnears:
    rainfall = []
    for month in months:
        print('month: '+month)
        msel_ds_rad = ds_rad.sel(time = month) 
        msel_ds_cmls = ds_cmls.sel(time = month) 
        # IDW merging initialization 
        merger = mergeplg.merge.MergeDifferenceIDW(
            ds_rad=msel_ds_rad.rainfall_amount,
            ds_cmls=msel_ds_cmls,
            method="additive",
            nnear=nnear,
            range_checks={'diff_check':diff_check_sel},
        )
        # merging application (loop on timesteps)
        for time in tqdm.tqdm(msel_ds_rad.time.data):
            rainfall.append(
                merger(
                    da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                    da_cmls=msel_ds_cmls.R_acc.sel(time=time),
                )
            )
    # Concat rainfall events
    data = xr.concat(rainfall, dim="time")
    
    # Overlapping timesteps in merging loop, remove them
    data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)
    
    # Fill missing timesteps with radar, or 0 if radar is nan
    data = data.fillna(ds_rad.rainfall_amount)
    data = data.fillna(0)

    compute_metrics(
        data,
        'data/metrics/OpenMRG_add_p_idw_nnear'+str(nnear)+'.csv', 
        'OpenMRG'
    )

    del data, merger

month: 2015-06


100%|█████████████████████████████████████████| 719/719 [00:11<00:00, 63.32it/s]


month: 2015-07


100%|█████████████████████████████████████████| 744/744 [00:10<00:00, 68.25it/s]


month: 2015-08


...

100%|█████████████████████████████████████████| 719/719 [00:21<00:00, 33.84it/s]


month: 2015-07


100%|█████████████████████████████████████████| 744/744 [00:22<00:00, 32.77it/s]


month: 2015-08


100%|█████████████████████████████████████████| 744/744 [00:22<00:00, 32.64it/s]


In [11]:
# multiplicative IDW  

for nnear in nnears:
    rainfall = []
    for month in months:
        print('month: '+month)
        msel_ds_rad = ds_rad.sel(time = month) 
        msel_ds_cmls = ds_cmls.sel(time = month) 
        # IDW merging initialization 
        merger = mergeplg.merge.MergeDifferenceIDW(
            ds_rad=msel_ds_rad.rainfall_amount,
            ds_cmls=msel_ds_cmls,
            method="multiplicative",
            nnear=nnear,
            range_checks={'ratio_check':ratio_check_sel},
            log_transform=True,
        )
        # merging application (loop on timesteps)
        for time in tqdm.tqdm(msel_ds_rad.time.data):
            rainfall.append(
                merger(
                    da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                    da_cmls=msel_ds_cmls.R_acc.sel(time=time),
                )
            )
    # Concat rainfall events
    data = xr.concat(rainfall, dim="time")
    
    # Overlapping timesteps in merging loop, remove them
    data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)
    
    # Fill missing timesteps with radar, or 0 if radar is nan
    data = data.fillna(0)

    compute_metrics(
        data,
        'data/metrics/OpenMRG_mul_p_idw_nnear'+str(nnear)+'.csv', 
        'OpenMRG'
    )

    del data, merger

month: 2015-06


100%|█████████████████████████████████████████| 719/719 [00:11<00:00, 64.22it/s]


month: 2015-07


100%|█████████████████████████████████████████| 744/744 [00:11<00:00, 65.01it/s]


month: 2015-08


...

100%|█████████████████████████████████████████| 719/719 [00:19<00:00, 36.71it/s]


month: 2015-07


100%|█████████████████████████████████████████| 744/744 [00:20<00:00, 36.45it/s]


month: 2015-08


100%|█████████████████████████████████████████| 744/744 [00:20<00:00, 36.94it/s]


In [12]:
# additive POINT ORDINARY KRIGING  
for nnear in nnears:
    variograms = estimate_several_variograms(ds_cmls, 'rainfall_difference')
    
    # Merge rainfall events for aggregated variograms
    rainfall = []
    ref_times = variograms.sort_index().index 
    for i in range(len(ref_times)-1):
        time_start = ref_times[i]
        time_end = ref_times[i +1]
        
        msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
        msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
        
        # OK merging initialization
        merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
            ds_rad=msel_ds_rad.rainfall_amount,
            ds_cmls=msel_ds_cmls,
            full_line=False,
            variogram_parameters={'nugget':0.3, 'psill':0.7, 'range': 30000},
            nnear=nnear,
            method="additive",
            range_checks={'diff_check':diff_check_sel},
        )
        # merging application (loop on timesteps)
        for time in tqdm.tqdm(msel_ds_rad.time.data):
            rainfall.append(
                merger(
                    da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                    da_cmls=msel_ds_cmls.R_acc.sel(time=time),
                )
            )
            
    # Concat rainfall events
    data = xr.concat(rainfall, dim="time")
    
    # Overlapping timesteps in merging loop, remove them
    data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)
    
    # Fill missing timesteps with radar, or 0 if radar is nan
    data = data.fillna(ds_rad.rainfall_amount)
    data = data.fillna(0)

    compute_metrics(
        data,
        'data/metrics/OpenMRG_add_p_ok_nnear'+str(nnear)+'.csv', 
        'OpenMRG'
    )
      
    del data, merger

100%|█████████████████████████████████████████| 168/168 [02:56<00:00,  1.05s/it]


In [13]:
# multiplicative POINT ORDINARY KRIGING  
for nnear in nnears:
    variograms = estimate_several_variograms(ds_cmls, 'rainfall_ratio')
    
    # Merge rainfall events for aggregated variograms
    rainfall = []
    ref_times = variograms.sort_index().index 
    for i in range(len(ref_times)-1):
        time_start = ref_times[i]
        time_end = ref_times[i +1]
        
        msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
        msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
        
        # OK merging initialization
        merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
            ds_rad=msel_ds_rad.rainfall_amount,
            ds_cmls=msel_ds_cmls,
            full_line=False,
            variogram_parameters={'nugget':0.3, 'psill':0.7, 'range': 30000},
            nnear=nnear,
            method="multiplicative",
            range_checks={'ratio_check':ratio_check_sel},
            log_transform=True,
        )
        # merging application (loop on timesteps)
        for time in tqdm.tqdm(msel_ds_rad.time.data):
            rainfall.append(
                merger(
                    da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                    da_cmls=msel_ds_cmls.R_acc.sel(time=time),
                )
            )
            
    # Concat rainfall events
    data = xr.concat(rainfall, dim="time")
    
    # Overlapping timesteps in merging loop, remove them
    data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)
    
    # Fill missing timesteps with radar, or 0 if radar is nan
    data = data.fillna(ds_rad.rainfall_amount)
    data = data.fillna(0)
    
    compute_metrics(
        data,
        'data/metrics/OpenMRG_mul_p_ok_nnear'+str(nnear) + '.csv', 
        'OpenMRG'
    )
    
    del data, merger

100%|█████████████████████████████████████████| 168/168 [02:08<00:00,  1.31it/s]


In [14]:
# additive BLOCK ORDINARY KRIGING 
for nnear in nnears:

    variograms = estimate_several_variograms(ds_cmls, 'rainfall_difference')
    
    # Merge rainfall events for aggregated variograms
    rainfall = []
    ref_times = variograms.sort_index().index 
    for i in range(len(ref_times)-1):
        time_start = ref_times[i]
        time_end = ref_times[i +1]
        
        msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
        msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
        
        # OK merging initialization
        merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
            ds_rad=msel_ds_rad.rainfall_amount,
            ds_cmls=msel_ds_cmls,
            full_line=True,
            variogram_parameters={'nugget':0.3, 'psill':0.7, 'range': 30000},
            nnear=nnear,
            method="additive",
            range_checks={'diff_check':diff_check_sel},
        )
        # merging application (loop on timesteps)
        for time in tqdm.tqdm(msel_ds_rad.time.data):
            rainfall.append(
                merger(
                    da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                    da_cmls=msel_ds_cmls.R_acc.sel(time=time),
                )
            )
            
    # Concat rainfall events
    data = xr.concat(rainfall, dim="time")
    
    # Overlapping timesteps in merging loop, remove them
    data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)
    
    # Fill missing timesteps with radar, or 0 if radar is nan
    data = data.fillna(ds_rad.rainfall_amount)
    data = data.fillna(0)
    
    compute_metrics(
        data,
        'data/metrics/OpenMRG_add_b_ok_nnear'+str(nnear)+'.csv', 
        'OpenMRG'
    )
  
    del data, merger

100%|█████████████████████████████████████████| 168/168 [03:09<00:00,  1.13s/it]


In [15]:
# multiplicative BLOCK ORDINARY KRIGING 
for nnear in nnears:

    variograms = estimate_several_variograms(ds_cmls, 'rainfall_ratio')
    
    # Merge rainfall events for aggregated variograms
    rainfall = []
    ref_times = variograms.sort_index().index 
    for i in range(len(ref_times)-1):
        time_start = ref_times[i]
        time_end = ref_times[i +1]
        
        msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
        msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
        
        # OK merging initialization
        merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
            ds_rad=msel_ds_rad.rainfall_amount,
            ds_cmls=msel_ds_cmls,
            full_line=True,
            variogram_parameters={'nugget':0.3, 'psill':0.7, 'range': 30000},
            nnear=nnear,
            method="multiplicative",
            range_checks={'ratio_check':ratio_check_sel},
            log_transform=True,
        )
        # merging application (loop on timesteps)
        for time in tqdm.tqdm(msel_ds_rad.time.data):
            rainfall.append(
                merger(
                    da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                    da_cmls=msel_ds_cmls.R_acc.sel(time=time),
                )
            )
            
    # Concat rainfall events
    data = xr.concat(rainfall, dim="time")
    
    # Overlapping timesteps in merging loop, remove them
    data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)
    
    # Fill missing timesteps with radar, or 0 if radar is nan
    data = data.fillna(ds_rad.rainfall_amount)
    data = data.fillna(0)
    
    compute_metrics(
        data,
        'data/metrics/OpenMRG_mul_b_ok_nnear'+str(nnear)+'.csv', 
        'OpenMRG'
    )

    del data, merger

100%|█████████████████████████████████████████| 168/168 [02:03<00:00,  1.36it/s]


In [16]:
# KED point 
for nnear in nnears:
    variograms = estimate_several_variograms(ds_cmls, 'rainfall_cml')
    
    # Merge rainfall events for aggregated variograms
    rainfall = []
    ref_times = variograms.sort_index().index 
    for i in range(len(ref_times)-1):
        time_start = ref_times[i]
        time_end = ref_times[i +1]
        
        msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
        msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
        
        # OK merging initialization
        merger = mergeplg.merge.MergeKrigingExternalDrift(
            ds_rad=ds_rad.rainfall_amount,
            ds_cmls=ds_cmls,
            variogram_parameters={'nugget':0.3, 'psill':0.7, 'range': 30000},
            nnear=nnear,
            full_line=False,
            range_checks={'diff_check':diff_check_sel},
        )
        # merging application (loop on timesteps)
        for time in tqdm.tqdm(msel_ds_rad.time.data):
            rainfall.append(
                merger(
                    da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                    da_cmls=msel_ds_cmls.R_acc.sel(time=time),
                )
            )
            
    # Concat rainfall events
    data = xr.concat(rainfall, dim="time")
    
    # Overlapping timesteps in merging loop, remove them
    data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)
    
    # Fill missing timesteps with radar, or 0 if radar is nan
    data = data.fillna(ds_rad.rainfall_amount)
    data = data.fillna(0)
    
    compute_metrics(
        data,
        'data/metrics/OpenMRG_ked_p_nnear'+str(nnear)+'.csv', 
        'OpenMRG'
    )
 
    del data, merger

100%|█████████████████████████████████████████| 168/168 [03:19<00:00,  1.19s/it]


In [17]:
# KED block  

for nnear in nnears:
    variograms = estimate_several_variograms(ds_cmls, 'rainfall_cml')
    
    # Merge rainfall events for aggregated variograms
    rainfall = []
    ref_times = variograms.sort_index().index 
    for i in range(len(ref_times)-1):
        time_start = ref_times[i]
        time_end = ref_times[i +1]
        
        msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
        msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
        
        # OK merging initialization
        merger = mergeplg.merge.MergeKrigingExternalDrift(
            ds_rad=ds_rad.rainfall_amount,
            ds_cmls=ds_cmls,
            variogram_parameters={'nugget':0.3, 'psill':0.7, 'range': 30000},
            nnear=nnear,
            full_line=True,
            range_checks={'diff_check':diff_check_sel},
        )
        # merging application (loop on timesteps)
        for time in tqdm.tqdm(msel_ds_rad.time.data):
            rainfall.append(
                merger(
                    da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                    da_cmls=msel_ds_cmls.R_acc.sel(time=time),
                )
            )
            
    # Concat rainfall events
    data = xr.concat(rainfall, dim="time")
    
    # Overlapping timesteps in merging loop, remove them
    data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)
    
    # Fill missing timesteps with radar, or 0 if radar is nan
    data = data.fillna(ds_rad.rainfall_amount)
    data = data.fillna(0)
    
    compute_metrics(
        data,
        'data/metrics/OpenMRG_ked_b_nnear'+str(nnear)+'.csv', 
        'OpenMRG'
    )
 
    del data, merger

100%|█████████████████████████████████████████| 168/168 [03:07<00:00,  1.12s/it]


# OpenRainER adjustment

In [18]:
# OpenRainER
ds_rad = xr.open_dataset("data/covi_2024_OpenRainER/openrainer_radar.nc")         
ds_cmls = xr.open_dataset("data/processed_cml_OpenRainER.nc")   
ds_gauges = xr.open_dataset('data/covi_2024_OpenRainER/AWS_rainfall.nc')        

# Get radar along CML, used for variogram estimation
da_intersect_weights = plg.spatial.calc_sparse_intersect_weights_for_several_cmls(
    x1_line=ds_cmls.site_0_lon.values,
    y1_line=ds_cmls.site_0_lat.values,
    x2_line=ds_cmls.site_1_lon.values,
    y2_line=ds_cmls.site_1_lat.values,
    cml_id=ds_cmls.cml_id.values,
    x_grid=ds_rad.lon.values,
    y_grid=ds_rad.lat.values,
    grid_point_location='center',
)
ds_cmls['radar_along_cml'] = plg.spatial.get_grid_time_series_at_intersections(
    grid_data=ds_rad.rainfall_amount,
    intersect_weights=da_intersect_weights,
)

# Difference used for additive
ds_cmls['rainfall_difference'] = xr.where(
    ds_cmls.radar_along_cml > 0, 
    ds_cmls.R_acc - ds_cmls.radar_along_cml, 
    np.nan
)

# Ratio used for multiplicative
ds_cmls['rainfall_ratio'] = xr.where(
    ds_cmls.radar_along_cml > 0, 
    ds_cmls.R_acc/ds_cmls.radar_along_cml, 
    np.nan
)

# CML obs used for KED
ds_cmls['rainfall_cml'] = xr.where(
    ds_cmls.radar_along_cml > 0, 
    ds_cmls.R_acc, 
    np.nan
)   

months = ['2022-06', '2022-07', '2022-08']

In [19]:
# methods parameter : default version
ratio_check_sel = (0.1,15)
diff_check_sel=10
nnears = [12, 70, 500] # 500 is more than all

In [20]:
# additive IDW 
for nnear in nnears:

    rainfall = []
    
    for month in months:
        print('month: '+month)
        msel_ds_rad = ds_rad.sel(time = month) 
        msel_ds_cmls = ds_cmls.sel(time = month) 
        # IDW merging initialization 
        merger = mergeplg.merge.MergeDifferenceIDW(
            ds_rad=msel_ds_rad.rainfall_amount,
            ds_cmls=msel_ds_cmls,
            method="additive",
            nnear=nnear,
            range_checks={'diff_check':diff_check_sel},
        )
        # merging application (loop on timesteps)
        for time in tqdm.tqdm(msel_ds_rad.time.data):
            rainfall.append(
                merger(
                    da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                    da_cmls=msel_ds_cmls.sel(time=time).R_acc,
                )
            )
    # Concat rainfall events
    data = xr.concat(rainfall, dim="time")
    
    # Overlapping timesteps in merging loop, remove them
    data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)
    
    # Fill missing timesteps with radar, or 0 if radar is nan
    data = data.fillna(ds_rad.rainfall_amount)
    data = data.fillna(0)
    
    compute_metrics(
        data,
        'data/metrics/OpenRainER_add_p_idw_nnear'+str(nnear)+'.csv', 
        'OpenRainER'
    )
     
    del data, merger

month: 2022-06


100%|█████████████████████████████████████████| 719/719 [00:48<00:00, 14.80it/s]


month: 2022-07


100%|█████████████████████████████████████████| 744/744 [00:47<00:00, 15.81it/s]


month: 2022-08


...

100%|█████████████████████████████████████████| 719/719 [03:14<00:00,  3.69it/s]


month: 2022-07


100%|█████████████████████████████████████████| 744/744 [03:08<00:00,  3.94it/s]


month: 2022-08


100%|█████████████████████████████████████████| 744/744 [03:10<00:00,  3.90it/s]


In [21]:
# multiplicative IDW 
for nnear in nnears:

    rainfall = []
    
    for month in months:
        print('month: '+month)
        msel_ds_rad = ds_rad.sel(time = month) 
        msel_ds_cmls = ds_cmls.sel(time = month) 
        # IDW merging initialization 
        merger = mergeplg.merge.MergeDifferenceIDW(
            ds_rad=msel_ds_rad.rainfall_amount,
            ds_cmls=msel_ds_cmls,
            method="multiplicative",
            nnear=nnear,
            range_checks={'ratio_check':ratio_check_sel},
            log_transform=True,
        )
        # merging application (loop on timesteps)
        for time in tqdm.tqdm(msel_ds_rad.time.data):
            rainfall.append(
                merger(
                    da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                    da_cmls=msel_ds_cmls.sel(time=time).R_acc,
                )
            )
    # Concat rainfall events
    data = xr.concat(rainfall, dim="time")
    
    # Overlapping timesteps in merging loop, remove them
    data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)
    
    # Fill missing timesteps with radar, or 0 if radar is nan
    data = data.fillna(ds_rad.rainfall_amount)
    data = data.fillna(0)
    
    compute_metrics(
        data,
        'data/metrics/OpenRainER_mul_p_idw_nnear'+str(nnear)+'.csv', 
        'OpenRainER'
    )
 
    del data, merger

month: 2022-06


100%|█████████████████████████████████████████| 719/719 [00:47<00:00, 15.17it/s]


month: 2022-07


100%|█████████████████████████████████████████| 744/744 [00:46<00:00, 15.85it/s]


month: 2022-08


...

100%|█████████████████████████████████████████| 719/719 [03:06<00:00,  3.85it/s]


month: 2022-07


100%|█████████████████████████████████████████| 744/744 [03:05<00:00,  4.02it/s]


month: 2022-08


100%|█████████████████████████████████████████| 744/744 [03:05<00:00,  4.02it/s]


In [22]:
# additive POINT ORDINARY KRIGING 
for nnear in nnears:

    variograms = estimate_several_variograms(ds_cmls, 'rainfall_difference')
    
    # Merge rainfall events for aggregated variograms
    rainfall = []
    ref_times = variograms.sort_index().index 
    for i in range(len(ref_times)-1):
        time_start = ref_times[i]
        time_end = ref_times[i +1]
        
        msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
        msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
        
        # OK merging initialization
        merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
            ds_rad=msel_ds_rad.rainfall_amount,
            ds_cmls=msel_ds_cmls,
            full_line=False,
            variogram_parameters={'nugget':0.3, 'psill':0.7, 'range': 30000},
            nnear=nnear,
            method="additive",
            range_checks={'diff_check':diff_check_sel},
        )
        # merging application (loop on timesteps)
        for time in tqdm.tqdm(msel_ds_rad.time.data):
            rainfall.append(
                merger(
                    da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                    da_cmls=msel_ds_cmls.R_acc.sel(time=time),
                )
            )
            
    # Concat rainfall events
    data = xr.concat(rainfall, dim="time")
    
    # Overlapping timesteps in merging loop, remove them
    data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)
    
    # Fill missing timesteps with radar, or 0 if radar is nan
    data = data.fillna(ds_rad.rainfall_amount)
    data = data.fillna(0)
    
    compute_metrics(
        data,
        'data/metrics/OpenRainER_add_p_ok_nnear'+str(nnear)+'.csv', 
        'OpenRainER'
    )
       
    del data, merger

100%|█████████████████████████████████████████| 264/264 [03:20<00:00,  1.32it/s]


In [23]:
# multiplicative POINT ORDINARY KRIGING 
for nnear in nnears:

    variograms = estimate_several_variograms(ds_cmls, 'rainfall_ratio')
    
    # Merge rainfall events for aggregated variograms
    rainfall = []
    ref_times = variograms.sort_index().index 
    for i in range(len(ref_times)-1):
        time_start = ref_times[i]
        time_end = ref_times[i +1]
        
        msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
        msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
        
        # OK merging initialization
        merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
            ds_rad=msel_ds_rad.rainfall_amount,
            ds_cmls=msel_ds_cmls,
            full_line=False,
            variogram_parameters={'nugget':0.3, 'psill':0.7, 'range': 30000},
            nnear=nnear,
            method="multiplicative",
            range_checks={'ratio_check':ratio_check_sel},
            log_transform=True,
        )
        # merging application (loop on timesteps)
        for time in tqdm.tqdm(msel_ds_rad.time.data):
            rainfall.append(
                merger(
                    da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                    da_cmls=msel_ds_cmls.R_acc.sel(time=time),
                )
            )
            
    # Concat rainfall events
    data = xr.concat(rainfall, dim="time")
    
    # Overlapping timesteps in merging loop, remove them
    data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)
    
    # Fill missing timesteps with radar, or 0 if radar is nan
    data = data.fillna(ds_rad.rainfall_amount)
    data = data.fillna(0)
    
    compute_metrics(
        data,
        'data/metrics/OpenRainER_mul_p_ok_nnear'+str(nnear)+'.csv', 
        'OpenRainER'
    )
     
    del data, merger

100%|█████████████████████████████████████████| 264/264 [02:04<00:00,  2.12it/s]


In [24]:
# additive BLOCK ORDINARY KRIGING  
for nnear in nnears:

    variograms = estimate_several_variograms(ds_cmls, 'rainfall_difference')
    
    # Merge rainfall events for aggregated variograms
    rainfall = []
    ref_times = variograms.sort_index().index 
    for i in range(len(ref_times)-1):
        time_start = ref_times[i]
        time_end = ref_times[i +1]
        
        msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
        msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
        
        # OK merging initialization
        merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
            ds_rad=msel_ds_rad.rainfall_amount,
            ds_cmls=msel_ds_cmls,
            full_line=True,
            variogram_parameters={'nugget':0.3, 'psill':0.7, 'range': 30000},
            nnear=nnear,
            method="additive",
            range_checks={'diff_check':diff_check_sel},
        )
        # merging application (loop on timesteps)
        for time in tqdm.tqdm(msel_ds_rad.time.data):
            rainfall.append(
                merger(
                    da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                    da_cmls=msel_ds_cmls.R_acc.sel(time=time),
                )
            )
            
    # Concat rainfall events
    data = xr.concat(rainfall, dim="time")
    
    # Overlapping timesteps in merging loop, remove them
    data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)
    
    # Fill missing timesteps with radar, or 0 if radar is nan
    data = data.fillna(ds_rad.rainfall_amount)
    data = data.fillna(0)
    
    compute_metrics(
        data,
        'data/metrics/OpenRainER_add_b_ok_nnear'+str(nnear)+'.csv', 
        'OpenRainER'
    )
     
    del data, merger

100%|█████████████████████████████████████████| 264/264 [03:15<00:00,  1.35it/s]


In [25]:
# multiplicative BLOCK ORDINARY KRIGING 
for nnear in nnears:

    variograms = estimate_several_variograms(ds_cmls, 'rainfall_ratio')
    
    # Merge rainfall events for aggregated variograms
    rainfall = []
    ref_times = variograms.sort_index().index 
    for i in range(len(ref_times)-1):
        time_start = ref_times[i]
        time_end = ref_times[i +1]
        
        msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
        msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
        
        # OK merging initialization
        merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
            ds_rad=msel_ds_rad.rainfall_amount,
            ds_cmls=msel_ds_cmls,
            full_line=True,
            variogram_parameters={'nugget':0.3, 'psill':0.7, 'range': 30000},
            nnear=nnear,
            method="multiplicative",
            range_checks={'ratio_check':ratio_check_sel},
            log_transform=True,
        )
        # merging application (loop on timesteps)
        for time in tqdm.tqdm(msel_ds_rad.time.data):
            rainfall.append(
                merger(
                    da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                    da_cmls=msel_ds_cmls.R_acc.sel(time=time),
                )
            )
            
    # Concat rainfall events
    data = xr.concat(rainfall, dim="time")
    
    # Overlapping timesteps in merging loop, remove them
    data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)
    
    # Fill missing timesteps with radar, or 0 if radar is nan
    data = data.fillna(ds_rad.rainfall_amount)
    data = data.fillna(0)
    
    compute_metrics(
        data,
        'data/metrics/OpenRainER_mul_b_ok_nnear'+str(nnear)+'.csv', 
        'OpenRainER'
    )
  
    del data, merger

100%|█████████████████████████████████████████| 264/264 [02:02<00:00,  2.15it/s]


In [26]:
# KED point 
for nnear in nnears:

    variograms = estimate_several_variograms(ds_cmls, 'rainfall_cml')
    
    # Merge rainfall events for aggregated variograms
    rainfall = []
    ref_times = variograms.sort_index().index 
    for i in range(len(ref_times)-1):
        time_start = ref_times[i]
        time_end = ref_times[i +1]
        
        msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
        msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
        
        # OK merging initialization
        merger = mergeplg.merge.MergeKrigingExternalDrift(
            ds_rad=ds_rad.rainfall_amount,
            ds_cmls=ds_cmls,
            variogram_parameters={'nugget':0.3, 'psill':0.7, 'range': 30000},
            nnear=nnear,
            full_line=False,
            range_checks={'diff_check':diff_check_sel},
        )
        # merging application (loop on timesteps)
        for time in tqdm.tqdm(msel_ds_rad.time.data):
            rainfall.append(
                merger(
                    da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                    da_cmls=msel_ds_cmls.R_acc.sel(time=time),
                )
            )
            
    # Concat rainfall events
    data = xr.concat(rainfall, dim="time")
    
    # Overlapping timesteps in merging loop, remove them
    data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)
    
    # Fill missing timesteps with radar, or 0 if radar is nan
    data = data.fillna(ds_rad.rainfall_amount)
    data = data.fillna(0)
    
    compute_metrics(
        data,
        'data/metrics/OpenRainER_ked_p_nnear'+str(nnear)+'.csv', 
        'OpenRainER'
    )
  
    del data, merger

100%|█████████████████████████████████████████| 264/264 [03:30<00:00,  1.25it/s]


In [27]:
# KED block  
for nnear in nnears:

    variograms = estimate_several_variograms(ds_cmls, 'rainfall_cml')
    
    # Merge rainfall events for aggregated variograms
    rainfall = []
    ref_times = variograms.sort_index().index 
    for i in range(len(ref_times)-1):
        time_start = ref_times[i]
        time_end = ref_times[i +1]
        
        msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
        msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
        
        # OK merging initialization
        merger = mergeplg.merge.MergeKrigingExternalDrift(
            ds_rad=ds_rad.rainfall_amount,
            ds_cmls=ds_cmls,
            variogram_parameters={'nugget':0.3, 'psill':0.7, 'range': 30000},
            nnear=nnear,
            full_line=True,
            range_checks={'diff_check':diff_check_sel},
        )
        # merging application (loop on timesteps)
        for time in tqdm.tqdm(msel_ds_rad.time.data):
            rainfall.append(
                merger(
                    da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                    da_cmls=msel_ds_cmls.R_acc.sel(time=time),
                )
            )
            
    # Concat rainfall events
    data = xr.concat(rainfall, dim="time")
    
    # Overlapping timesteps in merging loop, remove them
    data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)
    
    # Fill missing timesteps with radar, or 0 if radar is nan
    data = data.fillna(ds_rad.rainfall_amount)
    data = data.fillna(0)
    
    compute_metrics(
        data,
        'data/metrics/OpenRainER_ked_b_nnear'+str(nnear)+'.csv', 
        'OpenRainER'
    )

    del data, merger

100%|█████████████████████████████████████████| 264/264 [03:26<00:00,  1.28it/s]
